# Supplementary notebook 3. The four hypotheses tested (Fig. S8), with Figs. S5 and S6

Reproduces SI Figs. S5, S6 and S8 and SI Tables S6 and S7. Reads no data. Run time about 40 minutes. `QUICK = True` in the first code cell shortens the runs for a test.

In [1]:
QUICK = False
T_RUN = 3000      # the run read by Fig. 4 and the copying analysis of Fig. S2
T_POP = 5000      # the four conditions, the hypothesis tests, the sweeps and Fig. 5; the same seeds,
                  # so the first 3,000 steps of these runs are the runs of Fig. 4

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from math import log2

plt.rcParams.update({'figure.dpi': 110, 'font.size': 8, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.linewidth': 0.7,
                     'xtick.major.width': 0.7, 'ytick.major.width': 0.7,
                     'legend.frameon': False, 'font.family': 'DejaVu Sans'})

CEIL = log2(100)          # 6.6439 bits, the even-coupling ceiling at N = 100
BA_KAPPA = 10.10          # matched scale-free ensemble, control-entropy paper Table 1
BA_DH_DEG = 0.357         # its degree-weighted compression
BA_RDEG = -0.17           # its degree assortativity
N = 100
COL = {'baseline': '#7A7A7A', 'explore': '#2E7D4E', 'imitate': '#2E5090', 'both': '#A4161A'}
CONDS = {'baseline': dict(pr=0.0, pe=0.0), 'explore': dict(pr=0.0, pe=0.05),
         'imitate': dict(pr=0.3, pe=0.0), 'both': dict(pr=0.3, pe=0.05)}

def seeds(n):
    return list(range(n if not QUICK else max(2, n // 5)))

print('ceiling log2 N =', round(CEIL, 4), 'bits')

ceiling log2 N = 6.6439 bits


## Step 0. The model

In [2]:
import numpy as np
import networkx as nx

LOG2 = np.log(2.0)


# ----------------------------------------------------------------------
# measures (definitions of the second study)
# ----------------------------------------------------------------------
def control_entropy(v):
    """Control entropy in bits of a non-negative vector of shares (Eq. 11)."""
    v = np.asarray(v, float)
    tot = v.sum()
    if tot <= 0:
        return 0.0
    p = v / tot
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


def eigen_shares(W):
    """Shares from the leading eigenvector of a symmetric coupling matrix."""
    vals, vecs = np.linalg.eigh(W)
    v = np.abs(vecs[:, -1])
    return v / v.sum()


def adjacency_measures(A, W):
    """Measures read from the unweighted structure, computed as in Table 1 of the
    control-entropy paper: eigenvector weighting on the largest connected component,
    degree assortativity on the whole graph."""
    G = nx.from_numpy_array(A.astype(int))
    deg = A.sum(axis=1)
    s = W.sum(axis=1)
    iu, ju = np.nonzero(np.triu(A, 1))

    def edge_corr(v):
        x, y = np.concatenate([v[iu], v[ju]]), np.concatenate([v[ju], v[iu]])
        return float(np.corrcoef(x, y)[0, 1]) if x.std() > 1e-12 else float('nan')

    r_deg, r_str = edge_corr(deg.astype(float)), edge_corr(s)
    if not nx.is_connected(G):
        G = G.subgraph(max(nx.connected_components(G), key=len)).copy()
    c = nx.eigenvector_centrality_numpy(G)
    v = np.abs(np.array([c[n] for n in G.nodes()]))
    return dict(H_adj=control_entropy(v), r_deg=float(r_deg), r_str=r_str, gc=G.number_of_nodes())


# ----------------------------------------------------------------------
# the five rules, one function each
# ----------------------------------------------------------------------
def ring_lattice(N, k):
    """Regular ring lattice: every element coupled to its k nearest neighbours."""
    return nx.to_numpy_array(nx.watts_strogatz_graph(N, k, 0.0), dtype=bool)


def softmax_allocation(A, beta, shat):
    """Eq 1. Element i splits one unit of coupling over its partners j with weights
    exp(beta_i * shat_j). beta_i = 0 gives even coupling."""
    L = np.where(A, beta[:, None] * shat[None, :], -np.inf)
    L = L - L.max(axis=1, keepdims=True)
    E = np.where(A, np.exp(L), 0.0)
    return E / E.sum(axis=1, keepdims=True)


def commitment(phi0, phi1, C, fm):
    """Eq 2. Commitment of an element, in the linear form of the published model,
    f = phi0 + phi1 * C, truncated to [0, 1 - fm]."""
    return np.clip(phi0 + phi1 * C, 0.0, 1.0 - fm)


def protection(f, c, pmax, chalf):
    """Eq 8. Saturating protection bought by committed capability f * c."""
    x = f * c
    return np.where(x > 0, pmax / (1.0 + chalf / np.maximum(x, 1e-12)), 0.0)


def fermi(dc, s):
    """Eq 10. Probability of copying a partner whose capability is higher by dc."""
    return 1.0 / (1.0 + np.exp(-s * dc))


def logistic(z, g, theta):
    """Response function of an element, used inside the spread (Eq 5)."""
    return 1.0 / (1.0 + np.exp(-g * (z - theta)))


class ElementModel:
    """Network-agent model of element-level exploration and imitation.

    Each element holds
      beta_i        direction of coupling (0 = even coupling, > 0 = toward stronger partners)
      phi0_i, phi1_i commitment strategy, giving f_i = phi0_i + phi1_i * C_i
      c_i           capability, the accumulated stimulation it compares with partners
    Each coupling (i, j) holds a weight w_ij.
    """

    def __init__(self, N=100, k=6, seed=0,
                 pr=0.3, pe=0.05, sigma=0.1, sigma_f=0.01, s=2.0,   # strategy dynamics
                 M=10, rho=0.5, nE=5, switch=None, psi=0.3,        # feedback, two channels
                 gamma=1.0, pn=0.02, pl=0.3, lam=1.0, g=4.0, theta=1.0,   # seeding and spread
                 fm=0.1, pmax=1.0, chalf=0.1, f0=0.1,              # capability and protection
                 eta=0.05, delta=0.01, w0=1.0,                     # consolidation
                 pw=2.0, kmin=2, loss=1.0,                         # pruning and its cost
                 spread=True, two_part=True):
        self.p = dict(N=N, k=k, pr=pr, pe=pe, sigma=sigma, sigma_f=sigma_f, s=s, M=M, rho=rho,
                      nE=nE, switch=switch, psi=psi, gamma=gamma, pn=pn, pl=pl, lam=lam, g=g,
                      theta=theta, fm=fm, pmax=pmax, chalf=chalf, f0=f0, eta=eta, delta=delta,
                      w0=w0, pw=pw, kmin=kmin, loss=loss, spread=spread, two_part=two_part)
        self.rng = np.random.default_rng(seed)
        self.N, self.k = N, k
        self.A = ring_lattice(N, k)
        self.W = self.A.astype(float) * w0
        self.beta = np.zeros(N)
        self.phi0 = np.full(N, f0)
        self.phi1 = np.zeros(N)
        self.c = np.ones(N)
        self.x = np.zeros(N)          # activation of the previous step
        self.v = np.ones(N) / np.sqrt(N)   # leading eigenvector, kept warm
        self.t = 0
        self.n_act = 0.0
        self.n_rounds = 0.0
        self.set_essential(self.rng.choice(N, nE, replace=False))

    # ---------- state helpers ----------
    def set_essential(self, E):
        self.E = np.asarray(E)
        self.isE = np.zeros(self.N, bool)
        self.isE[self.E] = True

    def strengths(self):
        return self.W.sum(axis=1)

    def shat(self):
        s = self.strengths()
        return s / s.max()

    def centrality(self, iters=8):
        """Leading eigenvector of W by power iteration, warm started, scaled to a
        maximum of one. This is the C of the commitment rule (Eq 2)."""
        v = self.v
        for _ in range(iters):
            v = self.W @ v
            n = np.linalg.norm(v)
            if n < 1e-12:
                v = np.ones(self.N) / np.sqrt(self.N)
                break
            v = v / n
        self.v = np.abs(v)
        return self.v / max(self.v.max(), 1e-12)

    def allocation(self):
        return softmax_allocation(self.A, self.beta, self.shat())

    # ---------- one step ----------
    def feedback(self):
        """Eq 3. Two channels of information.
        Process feedback is intrinsic: every element receives afferent input in
        proportion to its own activation in the previous step.
        Result feedback is augmented: M packets, each landing on a task-essential
        element with probability rho and on any element otherwise."""
        p, rng = self.p, self.rng
        u = p['psi'] * self.x.copy()
        if p['M'] > 0:
            to_E = rng.random(p['M']) < p['rho']
            tgt = np.where(to_E, rng.choice(self.E, p['M']), rng.integers(0, self.N, p['M']))
            u = u + np.bincount(tgt, minlength=self.N).astype(float)
        return u

    def spread(self, u, a):
        """Eq 4 and Eq 5. Stimulus is seeded where information arrives and then
        travels along the couplings.
        Seeding: element i is stimulated with probability 1 - (1 - p_n) exp(-gamma u_i),
        so an element fires spontaneously at rate p_n and more often when information
        reaches it.
        Spread: an element already stimulated passes activation to partner j with
        the coupling it allocates, and j responds with probability
        pl * logistic(g (d_j - theta)). Each element activates at most once per step.
        The cascade is run to completion inside the step."""
        p, rng, N = self.p, self.rng, self.N
        act = rng.random(N) < (1.0 - (1.0 - p['pn']) * np.exp(-p['gamma'] * u))   # Eq 4
        if not p['spread']:
            self.n_act, self.n_rounds = float(act.sum()), 0.0
            return act
        frontier = act.copy()
        rounds = 0
        while frontier.any() and rounds < 20:
            # drive reaching each candidate through the allocation of active elements
            d = p['lam'] * self.k * (frontier @ a)
            prob = p['pl'] * logistic(d, p['g'], p['theta'])       # Eq 5
            new = (rng.random(N) < prob) & ~act
            act |= new
            frontier = new
            rounds += 1
        self.n_act, self.n_rounds = float(act.sum()), float(rounds)
        return act

    def step(self):
        p, rng, N = self.p, self.rng, self.N
        if p['switch'] and self.t > 0 and self.t % p['switch'] == 0:
            self.set_essential(rng.choice(N, p['nE'], replace=False))

        C = self.centrality()
        a = self.allocation()                                     # Eq 1
        f = commitment(self.phi0, self.phi1, C, p['fm']) if p['two_part'] else np.zeros(N)  # Eq 2
        u = self.feedback()                                       # Eq 3
        act = self.spread(u, a)                                   # Eq 4, Eq 5
        x = act.astype(float)
        self.x = x

        # Eq 6. Payoff is the stimulation the element gathers. It reaches partners
        # through its own allocation and it integrates in proportion to the coupling
        # it holds, so both the direction and the amount of coupling enter.
        s = self.strengths()
        pay = (s / s.mean()) * (a @ x)
        self.c = pay + np.clip(1.0 - p['fm'] - f, 0.0, None) * self.c   # Eq 7

        # Eq 8 and Eq 9. Commitment protects a coupling against decay.
        P = protection(f, self.c, p['pmax'], p['chalf'])
        Pij = 0.5 * (P[:, None] + P[None, :])
        abar = 0.5 * (a + a.T)
        self.W = np.where(self.A,
                          p['w0'] + (1 - p['delta'] * (1 - Pij)) * (self.W - p['w0'])
                          + p['eta'] * self.k * abar * np.outer(x, x), 0.0)

        # Eq S1. Long-term change: a coupling falls out of use, and unless the
        # capability committed to it holds it, it is lost and replaced elsewhere.
        n_rw = int(p['pw']) + (1 if rng.random() < p['pw'] - int(p['pw']) else 0)
        for _ in range(n_rw):
            self._rewire(P)

        # Eq 10. Imitation along a coupling, synchronous, two independent passes.
        b_old, f0_old, f1_old, c_old = self.beta.copy(), self.phi0.copy(), self.phi1.copy(), self.c.copy()
        for i in np.flatnonzero(rng.random(N) < p['pr']):
            nb = np.flatnonzero(self.A[i])
            j = nb[rng.integers(nb.size)]
            if rng.random() < fermi(c_old[j] - c_old[i], p['s']):
                self.beta[i] = b_old[j]
        if p['two_part']:
            for i in np.flatnonzero(rng.random(N) < p['pr']):
                nb = np.flatnonzero(self.A[i])
                j = nb[rng.integers(nb.size)]
                if rng.random() < fermi(c_old[j] - c_old[i], p['s']):
                    self.phi0[i], self.phi1[i] = f0_old[j], f1_old[j]

        # Eq S2. Exploration of both parts of the strategy.
        ex = rng.random(N) < p['pe']
        self.beta[ex] += rng.normal(0.0, p['sigma'], ex.sum())
        if p['two_part']:
            ex0 = rng.random(N) < p['pe']
            self.phi0[ex0] += rng.normal(0.0, p['sigma_f'], ex0.sum())
            ex1 = rng.random(N) < p['pe']
            self.phi1[ex1] += rng.normal(0.0, p['sigma_f'], ex1.sum())
        self.t += 1
        return x

    def _rewire(self, P):
        """One element drops its weakest coupling, unless protection holds it, and
        attaches to a new partner chosen with exp(beta_i * shat_j)."""
        p, rng = self.p, self.rng
        i = rng.integers(self.N)
        deg = self.A.sum(axis=1)
        nb = np.flatnonzero(self.A[i])
        nb = nb[deg[nb] > p['kmin']]
        if deg[i] <= p['kmin'] or nb.size == 0:
            return
        j = nb[np.argmin(self.W[i, nb])]
        if rng.random() < 0.5 * (P[i] + P[j]):     # the coupling is held
            return
        # the route is lost, and with it the capability the two elements had built
        # through it. With loss = 1 this is the capability loss of the published model.
        self.c[i] *= (1.0 - p['loss'])
        self.c[j] *= (1.0 - p['loss'])
        cand = np.flatnonzero(~self.A[i])
        cand = cand[cand != i]
        if cand.size == 0:
            return
        z = self.beta[i] * self.shat()[cand]
        w = np.exp(z - z.max())
        new = rng.choice(cand, p=w / w.sum())
        self.A[i, j] = self.A[j, i] = False
        self.W[i, j] = self.W[j, i] = 0.0
        self.A[i, new] = self.A[new, i] = True
        self.W[i, new] = self.W[new, i] = p['w0']

    # ---------- measures ----------
    def measures(self, adj=False):
        s = self.strengths()
        deg = self.A.sum(axis=1)
        top = np.argsort(s)[-len(self.E):]
        nbE = self.A[self.E].any(axis=0) & ~self.isE
        tot = s.sum()
        C = self.centrality()
        f = commitment(self.phi0, self.phi1, C, self.p['fm'])
        return dict(H_str=control_entropy(s), H_eig=control_entropy(eigen_shares(self.W)),
                    H_deg=control_entropy(deg), kappa=(deg ** 2).mean() / deg.mean(),
                    beta_mean=self.beta.mean(), beta_var=self.beta.var(),
                    f_mean=float(f.mean()), phi0_mean=float(self.phi0.mean()),
                    phi1_mean=float(self.phi1.mean()), c_mean=float(self.c.mean()),
                    act=self.n_act / self.N, rounds=self.n_rounds,
                    hub_align=self.isE[top].mean(),
                    share_E=s[self.isE].sum() / tot, share_nbE=s[nbE].sum() / tot,
                    **(adjacency_measures(self.A, self.W) if adj else {}))

    def run(self, T, every=25, adj=False):
        rec = {}
        for step in range(T):
            self.step()
            if step % every == 0 or step == T - 1:
                m = self.measures(adj=adj)
                m['t'] = self.t
                for key, v in m.items():
                    rec.setdefault(key, []).append(v)
        return {key: np.array(v) for key, v in rec.items()}


def replicate(T, seeds, adj=True, **kw):
    """Run the model for several seeds and return the final measures."""
    out = []
    for sd in seeds:
        m = ElementModel(seed=sd, **kw)
        for _ in range(T):
            m.step()
        out.append(m.measures(adj=adj))
    return {key: np.array([o[key] for o in out]) for key in out[0]}

In [3]:
from scipy.stats import mannwhitneyu

def cliffs_delta(a, b):
    a, b = np.asarray(a), np.asarray(b)
    gt = sum((x > y) for x in a for y in b)
    lt = sum((x < y) for x in a for y in b)
    return (gt - lt) / (len(a) * len(b))

def boot_median(a, reps=5000, seed=0):
    rng2 = np.random.default_rng(seed)
    a = np.asarray(a)
    d = [np.median(rng2.choice(a, len(a))) for _ in range(reps)]
    return np.percentile(d, [2.5, 97.5])

def boot_diff(a, b, reps=5000, seed=0):
    rng2 = np.random.default_rng(seed)
    a, b = np.asarray(a), np.asarray(b)
    d = [np.median(rng2.choice(a, len(a))) - np.median(rng2.choice(b, len(b))) for _ in range(reps)]
    return np.percentile(d, [2.5, 97.5])

def summary(tag, v, fmt='.3f'):
    v = np.asarray(v, float); lo, hi = boot_median(v)
    print(f'{tag}: median {np.median(v):{fmt}} [{lo:{fmt}}, {hi:{fmt}}], '
          f'mean {v.mean():{fmt}}, sd {v.std(ddof=1):{fmt}}, range {v.min():{fmt}} to {v.max():{fmt}}')

def compare(tag, a, b, unit=''):
    u, p = mannwhitneyu(a, b, alternative='two-sided')
    d = cliffs_delta(a, b)
    lo, hi = boot_diff(a, b)
    diff = np.median(a) - np.median(b)
    print(f'{tag}: U = {u:.1f}, p = {p:.4f}, Cliff delta = {d:+.2f}, '
          f'median difference = {diff:+.3f}{unit} [{lo:+.3f}, {hi:+.3f}]')
    return dict(U=u, p=p, d=d, diff=diff, lo=lo, hi=hi)

def star(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'

## Step 1. The factorial: exploration and imitation are both needed

In [4]:
NS = seeds(30)
FACT = {}
for name, kw in CONDS.items():
    FACT[name] = replicate(T_POP, NS, adj=True, **kw)
print(f'factorial, {len(NS)} seeds, {T_POP} steps')
for name in CONDS:
    r = FACT[name]
    print(f'--- {name}')
    summary('compression on strength shares (bits)', CEIL - r['H_str'])
    summary('degree heterogeneity kappa', r['kappa'], '.2f')
    summary('degree assortativity', r['r_deg'])
    summary('share on essential elements', r['share_E'])
    summary('hub alignment', r['hub_align'], '.2f')
    summary('mean direction preference', r['beta_mean'], '.2f')
    summary('mean commitment', r['f_mean'])
    print(f'    eigenvector weighting on the unweighted structure: compression {CEIL - r["H_adj"].mean():.3f} '
          f'+/- {r["H_adj"].std(ddof=1):.3f}, strength assortativity {np.nanmean(r["r_str"]):+.3f}')
v = CEIL - FACT['both']['H_str']
print(f'combined condition: {int((v > 1.0).sum())} of {len(v)} runs above 1 bit of compression, '
      f'{int((v < 0.6).sum())} below 0.6 bits')
print(f'paper 2 matched ensembles: scale-free kappa {BA_KAPPA}, degree compression {BA_DH_DEG}, r_deg {BA_RDEG}')

factorial, 30 seeds, 5000 steps
--- baseline
compression on strength shares (bits): median 0.631 [0.609, 0.641], mean 0.636, sd 0.057, range 0.557 to 0.809
degree heterogeneity kappa: median 9.73 [9.68, 9.95], mean 9.81, sd 0.34, range 9.26 to 10.57
degree assortativity: median -0.040 [-0.059, -0.013], mean -0.034, sd 0.052, range -0.140 to 0.091
share on essential elements: median 0.230 [0.222, 0.238], mean 0.233, sd 0.024, range 0.196 to 0.292
hub alignment: median 1.00 [0.80, 1.00], mean 0.88, sd 0.14, range 0.60 to 1.00
mean direction preference: median 0.00 [0.00, 0.00], mean 0.00, sd 0.00, range 0.00 to 0.00
mean commitment: median 0.100 [0.100, 0.100], mean 0.100, sd 0.000, range 0.100 to 0.100
    eigenvector weighting on the unweighted structure: compression 0.507 +/- 0.048, strength assortativity -0.015
--- explore
compression on strength shares (bits): median 0.881 [0.822, 0.954], mean 0.894, sd 0.135, range 0.647 to 1.250
degree heterogeneity kappa: median 10.27 [10.03, 10.

## Step 2. H1. Structural change requires variation and selection together

In [5]:
ST = {}
ST['H1a'] = compare('H1 kappa, both vs baseline', FACT['both']['kappa'], FACT['baseline']['kappa'])
ST['H1b'] = compare('H1 kappa, both vs exploration only', FACT['both']['kappa'], FACT['explore']['kappa'])
ST['H1c'] = compare('H1 essential share, both vs exploration only',
                    FACT['both']['share_E'], FACT['explore']['share_E'])
ST['H1d'] = compare('H1 degree assortativity, both vs exploration only',
                    FACT['both']['r_deg'], FACT['explore']['r_deg'])
ST['H1e'] = compare('H1 compression, both vs exploration only',
                    CEIL - FACT['both']['H_str'], CEIL - FACT['explore']['H_str'], ' bits')
ST['H1f'] = compare('H1 compression, imitation only vs baseline',
                    CEIL - FACT['imitate']['H_str'], CEIL - FACT['baseline']['H_str'], ' bits')

H1 kappa, both vs baseline: U = 776.0, p = 0.0000, Cliff delta = +0.72, median difference = +11.302 [+7.637, +13.303]
H1 kappa, both vs exploration only: U = 763.0, p = 0.0000, Cliff delta = +0.70, median difference = +10.768 [+7.097, +12.880]
H1 essential share, both vs exploration only: U = 739.0, p = 0.0000, Cliff delta = +0.64, median difference = +0.158 [+0.098, +0.180]
H1 degree assortativity, both vs exploration only: U = 80.0, p = 0.0000, Cliff delta = -0.82, median difference = -0.275 [-0.304, -0.217]
H1 compression, both vs exploration only: U = 555.0, p = 0.1224, Cliff delta = +0.23, median difference = +0.155 bits [-0.042, +0.291]
H1 compression, imitation only vs baseline: U = 491.0, p = 0.5493, Cliff delta = +0.09, median difference = +0.016 bits [-0.039, +0.041]


## Step 3. H2. The spread of stimulus is what ties payoff to position

In [6]:
pl_on = FACT['both']
pl_off = replicate(T_POP, NS, adj=True, spread=False)
print('--- spread off (seeding only)')
summary('compression on strength shares (bits)', CEIL - pl_off['H_str'])
summary('degree heterogeneity kappa', pl_off['kappa'], '.2f')
summary('share on essential elements', pl_off['share_E'])
summary('share of elements activated per step', pl_off['act'])
summary('share of elements activated per step, spread on', pl_on['act'])
ST['H2'] = compare('H2 essential share, spread on vs off', pl_on['share_E'], pl_off['share_E'])
ST['H2b'] = compare('H2 activated share, spread on vs off', pl_on['act'], pl_off['act'])
ST['H2c'] = compare('H2 kappa, spread on vs off', pl_on['kappa'], pl_off['kappa'])

--- spread off (seeding only)
compression on strength shares (bits): median 1.125 [1.050, 1.156], mean 1.092, sd 0.199, range 0.578 to 1.670
degree heterogeneity kappa: median 26.61 [24.81, 27.41], mean 25.86, sd 4.38, range 14.28 to 36.29
share on essential elements: median 0.426 [0.413, 0.439], mean 0.423, sd 0.044, range 0.307 to 0.545
share of elements activated per step: median 0.090 [0.085, 0.100], mean 0.095, sd 0.023, range 0.060 to 0.170
share of elements activated per step, spread on: median 0.155 [0.130, 0.210], mean 0.192, sd 0.106, range 0.090 to 0.550
H2 essential share, spread on vs off: U = 255.0, p = 0.0040, Cliff delta = -0.43, median difference = -0.035 [-0.088, -0.014]
H2 activated share, spread on vs off: U = 814.5, p = 0.0000, Cliff delta = +0.81, median difference = +0.065 [+0.035, +0.120]
H2 kappa, spread on vs off: U = 148.0, p = 0.0000, Cliff delta = -0.67, median difference = -5.570 [-9.522, -2.810]


## Step 4. H3. The two channels of feedback do different work

In [7]:
res_only = replicate(T_POP, NS, adj=True, psi=0.0)
pro_only = replicate(T_POP, NS, adj=True, M=0)
neither = replicate(T_POP, NS, adj=True, M=0, psi=0.0)
both_ch = FACT['both']
CHAN = {'both channels': both_ch, 'intrinsic only': pro_only, 'augmented only': res_only, 'neither': neither}
for nm, r in CHAN.items():
    print(f'--- {nm}')
    summary('compression on strength shares (bits)', CEIL - r['H_str'])
    summary('share on essential elements', r['share_E'])
    summary('hub alignment', r['hub_align'], '.2f')
    summary('share of elements activated per step', r['act'])
ST['H3a'] = compare('H3 essential share, augmented only vs intrinsic only',
                    res_only['share_E'], pro_only['share_E'])
ST['H3b'] = compare('H3 compression, both channels vs augmented only',
                    CEIL - both_ch['H_str'], CEIL - res_only['H_str'], ' bits')
ST['H3c'] = compare('H3 compression, intrinsic only vs neither',
                    CEIL - pro_only['H_str'], CEIL - neither['H_str'], ' bits')
ST['H3d'] = compare('H3 essential share, intrinsic only vs neither',
                    pro_only['share_E'], neither['share_E'])

--- both channels
compression on strength shares (bits): median 1.036 [0.849, 1.164], mean 0.950, sd 0.452, range 0.021 to 1.700
share on essential elements: median 0.391 [0.339, 0.410], mean 0.351, sd 0.118, range 0.067 to 0.502
hub alignment: median 1.00 [1.00, 1.00], mean 0.95, sd 0.10, range 0.60 to 1.00
share of elements activated per step: median 0.155 [0.130, 0.210], mean 0.192, sd 0.106, range 0.090 to 0.550
--- intrinsic only
compression on strength shares (bits): median 0.842 [0.775, 0.930], mean 0.761, sd 0.351, range 0.012 to 1.186
share on essential elements: median 0.037 [0.032, 0.044], mean 0.050, sd 0.050, range 0.019 to 0.239
hub alignment: median 0.00 [0.00, 0.00], mean 0.03, sd 0.09, range 0.00 to 0.40
share of elements activated per step: median 0.060 [0.050, 0.105], mean 0.108, sd 0.107, range 0.010 to 0.430
--- augmented only
compression on strength shares (bits): median 0.989 [0.889, 1.025], mean 0.872, sd 0.363, range 0.053 to 1.412
share on essential elements: 

## Step 5. H4. The search is solved where information and exploration meet

In [8]:
corner_both = replicate(T_POP, NS, adj=False, rho=1.0, pe=0.2, pr=0.3)
corner_info = replicate(T_POP, NS, adj=False, rho=1.0, pe=0.0, pr=0.3)
corner_expl = replicate(T_POP, NS, adj=False, rho=0.0, pe=0.2, pr=0.3)
for nm, r in [('both halves', corner_both), ('information alone', corner_info), ('exploration alone', corner_expl)]:
    print(f'--- {nm}')
    summary('compression on strength shares (bits)', CEIL - r['H_str'])
    summary('share on essential elements', r['share_E'])
ST['H4a'] = compare('H4 essential share, both halves vs information alone',
                    corner_both['share_E'], corner_info['share_E'])
ST['H4b'] = compare('H4 essential share, both halves vs exploration alone',
                    corner_both['share_E'], corner_expl['share_E'])
ST['H4c'] = compare('H4 compression, both halves vs information alone',
                    CEIL - corner_both['H_str'], CEIL - corner_info['H_str'], ' bits')

--- both halves
compression on strength shares (bits): median 1.306 [1.274, 1.382], mean 1.387, sd 0.451, range 0.093 to 3.083
share on essential elements: median 0.461 [0.453, 0.475], mean 0.458, sd 0.113, range 0.112 to 0.789
--- information alone
compression on strength shares (bits): median 0.822 [0.798, 0.837], mean 0.830, sd 0.068, range 0.687 to 1.004
share on essential elements: median 0.288 [0.275, 0.295], mean 0.289, sd 0.023, range 0.252 to 0.334
--- exploration alone
compression on strength shares (bits): median 0.957 [0.926, 1.082], mean 0.985, sd 0.306, range 0.003 to 1.560
share on essential elements: median 0.037 [0.034, 0.039], mean 0.040, sd 0.012, range 0.030 to 0.093
H4 essential share, both halves vs information alone: U = 813.0, p = 0.0000, Cliff delta = +0.81, median difference = +0.173 [+0.162, +0.195]
H4 essential share, both halves vs exploration alone: U = 900.0, p = 0.0000, Cliff delta = +1.00, median difference = +0.424 [+0.415, +0.439]
H4 compression, both

## Step 6. Figure S8. The four hypotheses in one figure

### The drawing style of the SI figures

In [9]:
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D

SI_RC = {'font.family': 'DejaVu Sans', 'font.size': 7, 'axes.labelsize': 6.6, 'xtick.labelsize': 6.0,
         'ytick.labelsize': 6.0, 'legend.fontsize': 6.0, 'axes.linewidth': 0.6, 'xtick.major.width': 0.6,
         'ytick.major.width': 0.6, 'xtick.major.size': 2.4, 'ytick.major.size': 0, 'legend.frameon': False,
         'axes.spines.top': True, 'axes.spines.right': True, 'mathtext.fontset': 'dejavusans'}
INK, NEUTRAL = '#1A1A1A', '#37474F'
BLUE, RED, GOLD, GREEN, GREY, NAVY = '#2E5090', '#A4161A', '#C9A227', '#2E7D4E', '#8A8A8A', '#1F3F7A'
SI_COND = {'baseline': '#C4C9CD', 'explore': '#8A8A8A', 'imitate': '#5F6B72', 'both': NAVY}
SI_LABEL = {'baseline': 'baseline', 'explore': 'exploration', 'imitate': 'imitation', 'both': 'both'}
_SI_HOT = plt.get_cmap('hot_r')(np.linspace(0.06, 0.94, 256))
_SI_COOL = plt.get_cmap('Blues')(np.linspace(0.92, 0.06, 256))
CMAP3 = LinearSegmentedColormap.from_list('hot_div', np.vstack([_SI_COOL, _SI_HOT]))
HOT3 = LinearSegmentedColormap.from_list('hot_half', CMAP3(np.linspace(0.50, 0.92, 256)))
SI_VLIM = 2.7157337518864986                       # the scale of Fig. 2 and of ref. 3, in bits
SI_CNORM = Normalize(-SI_VLIM, SI_VLIM)
DOT = (0, (1.2, 1.4))


def si_share_colour(w):
    # an element colored by its share against an even share, as in Figs. 2 and 3
    w = np.asarray(w, float); b = np.log2(w / w.sum() * len(w))
    return CMAP3(SI_CNORM(np.where(np.abs(b) < 1e-9, 0.0, b)))


def si_figure(w, h, label):
    plt.rcParams.update(SI_RC)
    fig = plt.figure(figsize=(w, h)); fig.set_label(label)
    return fig


def si_axes(fig, l, b, w, h):
    W, H = fig.get_size_inches()
    return fig.add_axes([l / W, b / H, w / W, h / H])


def si_head(fig, ax, letter, title, dy=0.12):
    # the title centered over the plot, the letter in bold beside its left end
    W, H = fig.get_size_inches()
    bb = ax.get_position()
    y = bb.y1 + dy / H
    t = fig.text((bb.x0 + bb.x1) / 2, y, title, fontsize=7.6, ha='center', va='bottom')
    ext = t.get_window_extent(fig.canvas.get_renderer()).transformed(fig.transFigure.inverted())
    fig.text(ext.x0 - 0.06 / W, y, letter, fontsize=9.0, fontweight='bold', ha='right', va='bottom')


def si_ticks(lo, hi):
    return [lo, (lo + hi) / 2, hi]


def si_nice(lo, hi, zero=False):
    # three round ticks, minimum, middle and maximum, as tight as round numbers allow around the data
    lo, hi = (min(0.0, lo), max(0.0, hi)) if zero else (lo, hi)
    span = (hi - lo) or abs(hi) or 1.0
    for e in range(int(np.floor(np.log10(span))) - 2, int(np.floor(np.log10(span))) + 2):
        for m in (1, 1.5, 2, 2.5, 3, 4, 5, 6, 8):
            u = m * 10.0 ** e
            a = np.floor(lo / u + 1e-9) * u
            b = np.ceil(hi / u - 1e-9) * u
            n = int(round((b - a) / u))
            if n % 2:
                b += u; n += 1
            if n <= 2:
                return [round(a, 10), round(a + u, 10), round(b, 10)]
    return [lo, (lo + hi) / 2, hi]


def si_frame(ax, xt=None, yt=None, xlog=False, ylog=False, xfmt=None, yfmt=None, grid=True):
    # a line above and below, none at the sides, ticks at the given values, a dotted rule at the middle tick
    if xlog:
        ax.set_xscale('log')
    if ylog:
        ax.set_yscale('log')
    g = lambda v, _: f'{v:,g}'
    if xt is not None:
        ax.set_xlim(xt[0], xt[-1]); ax.set_xticks(xt)
        ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(xfmt or g))
        ax.xaxis.set_minor_locator(mpl.ticker.NullLocator())
    if yt is not None:
        ax.set_ylim(yt[0], yt[-1]); ax.set_yticks(yt)
        ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(yfmt or g))
        ax.yaxis.set_minor_locator(mpl.ticker.NullLocator())
        if grid:
            for v in yt[1:-1]:
                ax.axhline(v, color='gray', alpha=0.45, ls=':', lw=0.5, zorder=0)
    for s_ in ('left', 'right'):
        ax.spines[s_].set_visible(False)
    for s_ in ('top', 'bottom'):
        ax.spines[s_].set_visible(True); ax.spines[s_].set_linewidth(0.7)
    ax.tick_params(axis='y', length=0, pad=1.4)
    ax.tick_params(axis='x', length=2.4, width=0.6, direction='out', pad=1.4)
    ax.set_axisbelow(True)


def si_cbar(fig, l, b, w, h, cmap, lo, hi, label, fmt='{:g}'):
    ax = si_axes(fig, l, b, w, h)
    ax.imshow(np.linspace(1, 0, 128)[:, None], cmap=cmap, aspect='auto', extent=[0, 1, lo, hi])
    ax.set_xticks([]); ax.set_yticks([lo, hi]); ax.set_yticklabels([fmt.format(lo), fmt.format(hi)])
    ax.yaxis.tick_right(); ax.yaxis.set_label_position('right')
    for s_ in ax.spines.values():
        s_.set_visible(True); s_.set_linewidth(0.5); s_.set_linestyle(DOT)
    ax.tick_params(axis='y', length=1.6, pad=1.0, labelsize=5.6)
    ax.set_ylabel(label, fontsize=6.0, labelpad=1.5)
    return ax


def si_bars(ax, groups, colors, labels, dots=True):
    # bars at the median of each group with every run as a dot beside it
    for i, v in enumerate(groups):
        v = np.asarray(v, float)
        ax.bar(i, np.median(v), color=colors[i], width=0.58, zorder=2)
        if dots:
            ax.plot(np.full(len(v), i) + 0.20 + np.linspace(-0.05, 0.05, len(v)), v, '.', ms=1.8, color=INK,
                    alpha=0.6, zorder=3)
    ax.set_xticks(range(len(groups))); ax.set_xticklabels(labels)
    ax.set_xlim(-0.6, len(groups) - 0.4)


def si_bracket(ax, i, j, y, h, text):
    ax.plot([i, i, j, j], [y, y + h, y + h, y], '-', lw=0.6, color=INK)
    ax.text((i + j) / 2, y + h, text, ha='center', va='bottom', fontsize=6.4)


def si_radial(A, n_min=2, dmax=None):
    # elements at the angle of their place on the ring, drawn inward by degree, as in Fig. 5C
    d = A.sum(1).astype(float); n = len(d)
    th = np.pi / 2 - 2 * np.pi * np.arange(n) / n
    dmax = max(d.max() if dmax is None else dmax, n_min + 1)
    r = np.clip(np.log(dmax / np.maximum(d, n_min)) / np.log(dmax / n_min), 0, 1)
    return np.c_[r * np.cos(th), r * np.sin(th)], d


def si_structure(ax, A, W=None, isE=None, s=None, lim=1.12, dmax=None):
    # a learned structure: couplings darker and wider with weight, elements colored by their share of the
    # coupling against an even share and sized by degree, the task-essential elements ringed
    from matplotlib.collections import LineCollection
    xy, d = si_radial(A, dmax=dmax)
    iu = np.array(np.where(np.triu(A))).T
    if W is not None:
        s = (W * A).sum(1) if s is None else s
        ww = W[iu[:, 0], iu[:, 1]]; x = np.sqrt((ww - ww.min()) / max(ww.max() - ww.min(), 1e-12))
    else:
        x = np.full(len(iu), 0.3)
    o = np.argsort(x, kind='stable')
    ax.add_collection(LineCollection(xy[iu[o]], colors=plt.get_cmap('Greys')(0.25 + 0.55 * x[o]),
                                     linewidths=0.12 + 0.8 * x[o], alpha=0.75, zorder=1))
    size = 3.0 + 110.0 * ((d - 1) / (len(d) - 2)) ** 1.15
    o = np.argsort(d, kind='stable')
    ax.scatter(xy[o, 0], xy[o, 1], s=size[o], c=si_share_colour(s)[o], edgecolors=NEUTRAL, linewidths=0.25, zorder=3)
    if isE is not None:
        e = np.where(isE)[0]
        ax.scatter(xy[e, 0], xy[e, 1], s=(np.sqrt(size[e]) + 2.6) ** 2, facecolors='none', edgecolors=INK,
                   linewidths=0.6, zorder=4)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect('equal'); ax.axis('off')

In [10]:
fig = si_figure(6.93, 1.95, 'SI Fig S8')
PW, PH = 1.18, 0.98
XS = [0.40, 2.12, 3.84, 5.56]

def head_room(ax, top, n):
    # the frame encloses the runs and the brackets above them
    yt = si_nice(0, top * (1.0 + 0.17 * n + 0.04), zero=True)
    si_frame(ax, None, yt)
    ax.tick_params(axis='x', length=0, labelsize=5.6)
    for lb in ax.get_xticklabels():
        lb.set_rotation(25); lb.set_ha('right')
    return top

ax = si_axes(fig, XS[0], 0.62, PW, PH)
g = [FACT[k]['kappa'] for k in CONDS]
si_bars(ax, g, [SI_COND[k] for k in CONDS], [SI_LABEL[k] for k in CONDS])
ax.axhline(BA_KAPPA, ls=':', color=INK, lw=0.7)
top = head_room(ax, max(v.max() for v in g), 2)
si_bracket(ax, 1, 3, top * 1.03, top * 0.04, star(ST['H1b']['p']))
si_bracket(ax, 0, 3, top * 1.19, top * 0.04, star(ST['H1a']['p']))
ax.set_ylabel(r'degree heterogeneity $\kappa$', labelpad=1)
si_head(fig, ax, 'A', 'H1')

ax = si_axes(fig, XS[1], 0.62, PW, PH)
g = [pl_off['act'], pl_on['act']]
si_bars(ax, g, ['#C4C9CD', GOLD], ['no spread', 'spread'])
top = head_room(ax, max(v.max() for v in g), 1)
si_bracket(ax, 0, 1, top * 1.03, top * 0.04, star(ST['H2b']['p']))
ax.set_ylabel('share activated per step', labelpad=1)
si_head(fig, ax, 'B', 'H2')

ax = si_axes(fig, XS[2], 0.62, PW, PH)
g = [pro_only['share_E'], res_only['share_E'], both_ch['share_E']]
si_bars(ax, g, [GREEN, GOLD, NAVY], ['intrinsic', 'augmented', 'both'])
top = head_room(ax, max(v.max() for v in g), 1)
si_bracket(ax, 0, 1, top * 1.03, top * 0.04, star(ST['H3a']['p']))
ax.set_ylabel('share on the essential set', labelpad=1)
si_head(fig, ax, 'C', 'H3')

ax = si_axes(fig, XS[3], 0.62, PW, PH)
g = [corner_info['share_E'], corner_expl['share_E'], corner_both['share_E']]
si_bars(ax, g, [GOLD, GREEN, NAVY], ['information', 'exploration', 'both'])
top = head_room(ax, max(v.max() for v in g), 2)
si_bracket(ax, 1, 2, top * 1.03, top * 0.04, star(ST['H4b']['p']))
si_bracket(ax, 0, 2, top * 1.19, top * 0.04, star(ST['H4a']['p']))
ax.set_ylabel('share on the essential set', labelpad=1)
si_head(fig, ax, 'D', 'H4')
plt.show()

## Step 7. How far stimulus travels (Fig. S5)

In [11]:
SW = seeds(10)
PL = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7]
pl_dH, pl_act, pl_kappa, pl_share = [], [], [], []
print(f'transmission probability sweep, {len(SW)} seeds per value, medians')
print('p_l     activation   dH       kappa   share_E')
for pl in PL:
    r = replicate(T_POP, SW, adj=True, pl=pl)
    pl_dH.append(np.median(CEIL - r['H_str'])); pl_act.append(np.median(r['act']))
    pl_kappa.append(np.median(r['kappa'])); pl_share.append(np.median(r['share_E']))
    print(f'p_l = {pl:<4} {pl_act[-1]:<12.3f} {pl_dH[-1]:.3f}    {pl_kappa[-1]:5.2f}   {pl_share[-1]:.3f}')


fig = si_figure(6.93, 1.62, 'SI Fig S5')
PW, PH = 1.62, 0.92
XS = [0.42, 2.78, 5.14]
XT = [0, 0.35, 0.7]
ax = si_axes(fig, XS[0], 0.40, PW, PH)
ax.plot(PL, pl_act, 'o-', ms=2.6, lw=0.9, color=GOLD)
si_frame(ax, XT, [0, 0.5, 1])
ax.set_xlabel(r'transmission probability $p_l$', labelpad=1); ax.set_ylabel('share activated per step', labelpad=1)
si_head(fig, ax, 'A', 'Activation')
ax = si_axes(fig, XS[1], 0.40, PW, PH)
ax.plot(PL, pl_dH, 'o-', ms=2.6, lw=0.9, color=RED)
si_frame(ax, XT, si_nice(0, max(pl_dH), zero=True))
ax.set_xlabel(r'transmission probability $p_l$', labelpad=1); ax.set_ylabel('bits saved', labelpad=1)
si_head(fig, ax, 'B', 'Compression')
ax = si_axes(fig, XS[2], 0.40, PW, PH)
ax.plot(PL, pl_kappa, 'o-', ms=2.6, lw=0.9, color=NAVY)
ax.axhline(BA_KAPPA, ls=':', color=INK, lw=0.7)
ax.text(0.7, BA_KAPPA + 0.6, 'scale-free', fontsize=5.6, ha='right', va='bottom', color=NEUTRAL)
si_frame(ax, XT, si_nice(0, max(pl_kappa), zero=True), grid=False)
ax.set_xlabel(r'transmission probability $p_l$', labelpad=1); ax.set_ylabel(r'degree heterogeneity $\kappa$', labelpad=1)
si_head(fig, ax, 'C', 'Hubs')
plt.show()

transmission probability sweep, 10 seeds per value, medians
p_l     activation   dH       kappa   share_E
p_l = 0.0  0.100        1.140    28.91   0.425
p_l = 0.1  0.105        1.052    23.96   0.420
p_l = 0.2  0.105        1.107    26.09   0.417
p_l = 0.3  0.150        1.040    22.54   0.391
p_l = 0.5  0.555        0.345     8.91   0.091
p_l = 0.7  0.835        0.284     8.77   0.057


## Step 8. Precision, a moving target, and transfer (Fig. S6)

In [12]:
print(f'feedback precision sweep, {len(SW)} seeds per value, medians')
RHO = [0.0, 0.25, 0.5, 0.75, 1.0]
rho_dH, rho_sh, rho_al = [], [], []
for rho in RHO:
    r = replicate(T_POP, SW, adj=False, rho=rho)
    rho_dH.append(np.median(CEIL - r['H_str'])); rho_sh.append(np.median(r['share_E']))
    rho_al.append(np.median(r['hub_align']))
    print(f'rho = {rho:<5} dH = {rho_dH[-1]:.3f}   share_E = {rho_sh[-1]:.3f}   hub alignment = {rho_al[-1]:.2f}')

print('moving target, medians')
SWI = [None, 1000, 250, 50]
sw_dH, sw_sh = [], []
for sv in SWI:
    r = replicate(T_POP, SW, adj=False, switch=sv)
    sw_dH.append(np.median(CEIL - r['H_str'])); sw_sh.append(np.median(r['share_E']))
    lab = 'fixed' if sv is None else str(sv)
    print(f'switch = {lab:<6} dH = {sw_dH[-1]:.3f}   share_E = {sw_sh[-1]:.3f}   '
          f'hub alignment = {np.median(r["hub_align"]):.2f}')

print('transfer between two tasks that share essential elements')
T_A, T_B = (T_POP, 1500) if not QUICK else (300, 300)
SHARED = [0, 1, 3, 5]
tr_curves = {}
for msh in SHARED:
    early, late = [], []
    for sd in SW:
        mm = ElementModel(seed=sd)
        for _ in range(T_A):
            mm.step()
        Ea = mm.E.copy()
        keep = Ea[:msh]
        pool = np.setdiff1d(np.arange(N), Ea)
        new = mm.rng.choice(pool, mm.p['nE'] - msh, replace=False)
        mm.set_essential(np.concatenate([keep, new]))
        al = []
        for step in range(T_B):
            mm.step()
            if step % 25 == 0:
                al.append(mm.measures()['hub_align'])
        al = np.array(al)
        early.append(al[:max(1, len(al) // 3)].mean()); late.append(al[-3:].mean())
    tr_curves[msh] = (np.median(early), np.median(late), np.std(early, ddof=1))
    print(f'{msh} shared: alignment over the first 500 steps = {np.median(early):.2f}, '
          f'at the end = {np.median(late):.2f}')

fig = si_figure(6.93, 1.70, 'SI Fig S6')
PW, PH = 1.18, 0.92
XS = [0.40, 2.12, 3.84, 5.56]
ax = si_axes(fig, XS[0], 0.48, PW, PH)
ks = list(CHAN)
lab = {'both channels': 'both', 'intrinsic only': 'intrinsic', 'augmented only': 'augmented', 'neither': 'neither'}
colk = {'both channels': NAVY, 'intrinsic only': GREEN, 'augmented only': GOLD, 'neither': '#C4C9CD'}
vals = [CEIL - CHAN[k]['H_str'] for k in ks]
si_bars(ax, vals, [colk.get(k, GREY) for k in ks], [lab.get(k, k) for k in ks])
si_frame(ax, None, si_nice(0, max(v.max() for v in vals), zero=True))
ax.tick_params(axis='x', length=0, labelsize=5.6)
for lb in ax.get_xticklabels():
    lb.set_rotation(25); lb.set_ha('right')
ax.set_ylabel('bits saved', labelpad=1)
si_head(fig, ax, 'A', 'Feedback channels')
ax = si_axes(fig, XS[1], 0.48, PW, PH)
ax.plot(RHO, rho_sh, 'o-', ms=2.6, lw=0.9, color=NAVY, label='essential share', clip_on=False, zorder=3)
ax.plot(RHO, rho_al, 's--', ms=2.4, lw=0.8, color=GREY, label='hub alignment', clip_on=False, zorder=3)
si_frame(ax, [0, 0.5, 1], [0, 0.5, 1])
ax.tick_params(axis='y', pad=3.2)
ax.legend(loc='lower right', handlelength=1.4, labelspacing=0.2, fontsize=5.4, borderaxespad=0.1)
ax.set_xlabel(r'feedback precision $\rho$', labelpad=1); ax.set_ylabel('share', labelpad=1)
si_head(fig, ax, 'B', 'Precision')
ax = si_axes(fig, XS[2], 0.48, PW, PH)
ax.bar(range(len(SWI)), sw_sh, color=[NAVY, BLUE, '#7F98C2', '#B9C6DD'], width=0.58)
ax.set_xticks(range(len(SWI))); ax.set_xticklabels(['fixed', '1,000', '250', '50'])
si_frame(ax, None, si_nice(0, max(sw_sh), zero=True))
ax.tick_params(axis='x', length=0)
ax.set_xlim(-0.6, len(SWI) - 0.4)
ax.set_xlabel('steps between target changes', labelpad=1); ax.set_ylabel('share on the essential set', labelpad=1)
si_head(fig, ax, 'C', 'A moving target')
ax = si_axes(fig, XS[3], 0.48, PW, PH)
ax.errorbar(SHARED, [tr_curves[k][0] for k in SHARED], yerr=[tr_curves[k][2] for k in SHARED], fmt='o-', ms=2.6,
            lw=0.9, color=GREEN, capsize=1.6, elinewidth=0.6, label='first 500 steps', clip_on=False, zorder=3)
ax.plot(SHARED, [tr_curves[k][1] for k in SHARED], 's--', ms=2.4, lw=0.8, color=NAVY, label='end', clip_on=False,
        zorder=3)
si_frame(ax, [0, 2.5, 5], [0, 0.5, 1], xfmt=lambda v, _: f'{v:g}')
ax.tick_params(axis='y', pad=3.2)
ax.legend(loc='upper left', handlelength=1.4, labelspacing=0.2, fontsize=5.4, borderaxespad=0.1)
ax.set_xlabel('shared essential elements', labelpad=1); ax.set_ylabel('alignment with task B', labelpad=1)
si_head(fig, ax, 'D', 'Transfer')
plt.show()

feedback precision sweep, 10 seeds per value, medians
rho = 0.0   dH = 0.648   share_E = 0.038   hub alignment = 0.00
rho = 0.25  dH = 1.035   share_E = 0.337   hub alignment = 0.80
rho = 0.5   dH = 1.040   share_E = 0.391   hub alignment = 1.00
rho = 0.75  dH = 0.976   share_E = 0.388   hub alignment = 1.00
rho = 1.0   dH = 1.180   share_E = 0.430   hub alignment = 1.00
moving target, medians
switch = fixed  dH = 1.040   share_E = 0.391   hub alignment = 1.00
switch = 1000   dH = 0.400   share_E = 0.103   hub alignment = 0.40
switch = 250    dH = 0.950   share_E = 0.067   hub alignment = 0.10
switch = 50     dH = 0.766   share_E = 0.055   hub alignment = 0.00
transfer between two tasks that share essential elements
0 shared: alignment over the first 500 steps = 0.04, at the end = 0.60
1 shared: alignment over the first 500 steps = 0.20, at the end = 0.60
3 shared: alignment over the first 500 steps = 0.60, at the end = 0.70
5 shared: alignment over the first 500 steps = 1.00, at the e

## Step 9. Selection intensity, and the consolidation settings of the scale-free paper

In [13]:
print(f'selection intensity sweep, {len(SW)} seeds per value, medians')
print('s        dH        mean beta   mean f')
for sv in [0.0, 0.5, 1.0, 2.0, 4.0, 8.0]:
    r = replicate(T_POP, SW, adj=False, s=sv)
    print(f's = {sv:<6} dH = {np.median(CEIL - r["H_str"]):.3f}   mean beta = {np.median(r["beta_mean"]):+.2f}   '
          f'mean f = {np.median(r["f_mean"]):.3f}')

print('Hebbian rate 0.01 and relaxation 0.001, the settings of the scale-free paper, medians')
print('condition       dH strength   kappa   share_E')
hb = {}
for name, kw in CONDS.items():
    r = replicate(T_POP, SW, adj=True, eta=0.01, delta=0.001, **kw)
    hb[name] = np.median(CEIL - r['H_str'])
    print(f'{name:<12} {hb[name]:.3f}         {np.median(r["kappa"]):5.2f}   {np.median(r["share_E"]):.3f}')
print('combined condition highest under the parent settings:',
      hb['both'] > max(hb[k] for k in ['baseline', 'explore', 'imitate']))

selection intensity sweep, 10 seeds per value, medians
s        dH        mean beta   mean f
s = 0.0    dH = 0.536   mean beta = +0.04   mean f = 0.058
s = 0.5    dH = 1.011   mean beta = +2.38   mean f = 0.019
s = 1.0    dH = 0.900   mean beta = +1.49   mean f = 0.001
s = 2.0    dH = 1.040   mean beta = +2.12   mean f = 0.000
s = 4.0    dH = 1.145   mean beta = +2.78   mean f = 0.000
s = 8.0    dH = 1.080   mean beta = +2.41   mean f = 0.000
Hebbian rate 0.01 and relaxation 0.001, the settings of the scale-free paper, medians
condition       dH strength   kappa   share_E
baseline     0.327          8.22   0.171
explore      0.464          8.92   0.173
imitate      0.328          8.03   0.168
both         0.961         17.89   0.361
combined condition highest under the parent settings: True


## Step 10. Checks

In [14]:
ok = {}
med = lambda k, key: np.median(FACT[k][key])
ok['combined condition above the other three, compression'] = med('both', 'H_str') < min(med(k, 'H_str') for k in ['baseline', 'explore', 'imitate'])
ok['imitation alone equals baseline, compression within 0.1 bits'] = abs(med('imitate', 'H_str') - med('baseline', 'H_str')) < 0.1
ok['combined condition most heterogeneous'] = med('both', 'kappa') > max(med(k, 'kappa') for k in ['baseline', 'explore', 'imitate'])
ok['combined condition most concentrated on essential elements'] = med('both', 'share_E') > max(med(k, 'share_E') for k in ['baseline', 'explore', 'imitate'])
ok['combined condition disassortative, the others not'] = med('both', 'r_deg') < -0.1 and all(med(k, 'r_deg') > -0.1 for k in ['baseline', 'explore', 'imitate'])
ok['H1 essential share separates both from exploration alone (p < 0.05)'] = ST['H1c']['p'] < 0.05
ok['H2 activated share separates spread on from off (p < 0.05)'] = ST['H2b']['p'] < 0.05
ok['H3 augmented channel places the change (p < 0.05)'] = ST['H3a']['p'] < 0.05
ok['H4 both halves beat either alone (p < 0.05)'] = ST['H4a']['p'] < 0.05 and ST['H4b']['p'] < 0.05
ok['commitment is learned as a rule on centrality'] = np.median(FACT['both']['phi1_mean']) > np.median(FACT['baseline']['phi1_mean'])
for kk, vv in ok.items():
    print(f'{"PASS" if vv else "FAIL"}  {kk}')
print('all checks passed:', all(ok.values()))

PASS  combined condition above the other three, compression
PASS  imitation alone equals baseline, compression within 0.1 bits
PASS  combined condition most heterogeneous
PASS  combined condition most concentrated on essential elements
PASS  combined condition disassortative, the others not
PASS  H1 essential share separates both from exploration alone (p < 0.05)
PASS  H2 activated share separates spread on from off (p < 0.05)
PASS  H3 augmented channel places the change (p < 0.05)
PASS  H4 both halves beat either alone (p < 0.05)
FAIL  commitment is learned as a rule on centrality
all checks passed: False
